In [1]:
from torchvision.models import efficientnet_b0, efficientnet_b1, efficientnet_b2, efficientnet_b3, efficientnet_b7
from torchvision.models import EfficientNet_B0_Weights, EfficientNet_B1_Weights, EfficientNet_B2_Weights, EfficientNet_B3_Weights, EfficientNet_B7_Weights
import torch
import random
import numpy as np
import pandas as pd
from torch.utils.data.sampler import RandomSampler
import os
from utils.dataset import PandasDataset
from utils.metrics import evaluation, format_metrics
from utils.models import EfficientNetApi, EnsembleEfficientNet

In [2]:
seed = 42
batch_size = 6
num_workers = 4
output_classes = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

ROOT_DIR = '../../..'
data_dir = '../../../../dataset'
images_dir = os.path.join(data_dir, 'tiles')

Using device: cuda


## Load Pre-trained Models

In [3]:
# Load EfficientNet-B0
load_model_b0 = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
model_b0 = EfficientNetApi(model=load_model_b0, output_dimensions=output_classes, dropout_rate=0.6)
model_b0.load_state_dict(torch.load("models/b0-entropy.pth"))
model_b0 = model_b0.to(device)
print("✓ Loaded EfficientNet-B0")

✓ Loaded EfficientNet-B0


In [4]:
# Load EfficientNet-B1
load_model_b1 = efficientnet_b1(weights=EfficientNet_B1_Weights.DEFAULT)
model_b1 = EfficientNetApi(model=load_model_b1, output_dimensions=output_classes, dropout_rate=0.6)
model_b1.load_state_dict(torch.load("models/b1.pth"))
model_b1 = model_b1.to(device)
print("✓ Loaded EfficientNet-B1")

✓ Loaded EfficientNet-B1


In [5]:
# Load EfficientNet-B2
load_model_b2 = efficientnet_b2(weights=EfficientNet_B2_Weights.DEFAULT)
model_b2 = EfficientNetApi(model=load_model_b2, output_dimensions=output_classes, dropout_rate=0.6)
model_b2.load_state_dict(torch.load("models/b2.pth"))
model_b2 = model_b2.to(device)
print("✓ Loaded EfficientNet-B2")

✓ Loaded EfficientNet-B2


In [6]:
# Load EfficientNet-B3
load_model_b3 = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
model_b3 = EfficientNetApi(model=load_model_b3, output_dimensions=output_classes, dropout_rate=0.6)
model_b3.load_state_dict(torch.load("models/b3.pth"))
model_b3 = model_b3.to(device)
print("✓ Loaded EfficientNet-B3")

✓ Loaded EfficientNet-B3


In [7]:
# Load EfficientNet-B7
load_model_b7 = efficientnet_b7(weights=EfficientNet_B7_Weights.DEFAULT)
model_b7 = EfficientNetApi(model=load_model_b7, output_dimensions=output_classes, dropout_rate=0.6)
model_b7.load_state_dict(torch.load("models/b7.pth"))
model_b7 = model_b7.to(device)
print("✓ Loaded EfficientNet-B7")

✓ Loaded EfficientNet-B7


## Load Test Dataset

In [8]:
df_test = pd.read_csv(f"{ROOT_DIR}/data/test.csv")
test_dataset = PandasDataset(images_dir, df_test, transforms=None)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=batch_size, num_workers=num_workers, sampler=RandomSampler(test_dataset)
)
print(f"Test dataset size: {len(test_dataset)}")

Test dataset size: 1592


## Create and Evaluate Ensemble Models

We'll test all available ensemble methods:
- `max`: Maximum of probabilities
- `mean`: Simple average of probabilities
- `weighted_mean`: Weighted average of probabilities
- `majority_vote`: Hard voting (most common prediction)
- `weighted_vote`: Soft voting with weights

In [9]:
# List of models for ensemble
# models_list = [model_b0, model_b1, model_b2, model_b3]
models_list = [model_b0, model_b3, model_b7]
print(f"Ensemble with {len(models_list)} models: B0, B3, B7")

Ensemble with 3 models: B0, B3, B7


### 1. Max Method

In [10]:
ensemble_max = EnsembleEfficientNet(models=models_list, method='max')
ensemble_max = ensemble_max.to(device)

print("Evaluating ensemble with 'max' method...")
response_max = evaluation(ensemble_max, test_loader, device)
result_max = format_metrics(response_max[0])
print("\n=== MAX METHOD ===")
print(result_max)

Evaluating ensemble with 'max' method...


100%|██████████| 266/266 [09:28<00:00,  2.14s/it]



=== MAX METHOD ===
VAL_ACC      Mean: 58.388 | Std: 1.243 | 95% CI: [56.344, 60.427]
VAL_KAPPA    Mean: 0.819 | Std: 0.012 | 95% CI: [0.799, 0.839]
VAL_F1       Mean: 0.533 | Std: 0.013 | 95% CI: [0.513, 0.554]
VAL_RECALL   Mean: 0.546 | Std: 0.013 | 95% CI: [0.525, 0.567]
VAL_PRECISION Mean: 0.533 | Std: 0.013 | 95% CI: [0.513, 0.553]


### 2. Mean Method

In [11]:
ensemble_mean = EnsembleEfficientNet(models=models_list, method='mean')
ensemble_mean = ensemble_mean.to(device)

print("Evaluating ensemble with 'mean' method...")
response_mean = evaluation(ensemble_mean, test_loader, device)
result_mean = format_metrics(response_mean[0])
print("\n=== MEAN METHOD ===")
print(result_mean)

Evaluating ensemble with 'mean' method...


100%|██████████| 266/266 [09:26<00:00,  2.13s/it]



=== MEAN METHOD ===
VAL_ACC      Mean: 64.935 | Std: 1.230 | 95% CI: [62.814, 66.897]
VAL_KAPPA    Mean: 0.843 | Std: 0.012 | 95% CI: [0.822, 0.863]
VAL_F1       Mean: 0.586 | Std: 0.013 | 95% CI: [0.564, 0.608]
VAL_RECALL   Mean: 0.585 | Std: 0.013 | 95% CI: [0.564, 0.606]
VAL_PRECISION Mean: 0.591 | Std: 0.014 | 95% CI: [0.569, 0.614]


### 3. Weighted Mean Method

Using weights based on model complexity (higher weight for larger models)

In [17]:
# Weights based on model size/performance (can be adjusted)
weights = [0.4, 0.3, 0.3]  # B0, B1, B2, B3

ensemble_weighted_mean = EnsembleEfficientNet(models=models_list, method='weighted_mean', weights=weights)
ensemble_weighted_mean = ensemble_weighted_mean.to(device)

print(f"Evaluating ensemble with 'weighted_mean' method (weights: {weights})...")
response_weighted_mean = evaluation(ensemble_weighted_mean, test_loader, device)
result_weighted_mean = format_metrics(response_weighted_mean[0])
print("\n=== WEIGHTED MEAN METHOD ===")
print(result_weighted_mean)

Evaluating ensemble with 'weighted_mean' method (weights: [0.4, 0.3, 0.3])...


100%|██████████| 266/266 [09:33<00:00,  2.15s/it]



=== WEIGHTED MEAN METHOD ===
VAL_ACC      Mean: 64.643 | Std: 1.131 | 95% CI: [62.751, 66.520]
VAL_KAPPA    Mean: 0.842 | Std: 0.011 | 95% CI: [0.824, 0.860]
VAL_F1       Mean: 0.584 | Std: 0.012 | 95% CI: [0.563, 0.603]
VAL_RECALL   Mean: 0.584 | Std: 0.012 | 95% CI: [0.564, 0.603]
VAL_PRECISION Mean: 0.589 | Std: 0.013 | 95% CI: [0.568, 0.610]


### 4. Majority Vote Method

In [13]:
ensemble_majority = EnsembleEfficientNet(models=models_list, method='majority_vote')
ensemble_majority = ensemble_majority.to(device)

print("Evaluating ensemble with 'majority_vote' method...")
response_majority = evaluation(ensemble_majority, test_loader, device)
result_majority = format_metrics(response_majority[0])
print("\n=== MAJORITY VOTE METHOD ===")
print(result_majority)

Evaluating ensemble with 'majority_vote' method...


100%|██████████| 266/266 [09:24<00:00,  2.12s/it]



=== MAJORITY VOTE METHOD ===
VAL_ACC      Mean: 11.693 | Std: 0.792 | 95% CI: [10.364, 13.006]
VAL_KAPPA    Mean: 0.000 | Std: 0.000 | 95% CI: [0.000, 0.000]
VAL_F1       Mean: 0.035 | Std: 0.002 | 95% CI: [0.031, 0.038]
VAL_RECALL   Mean: 0.167 | Std: 0.000 | 95% CI: [0.167, 0.167]
VAL_PRECISION Mean: 0.019 | Std: 0.001 | 95% CI: [0.017, 0.022]


### 5. Weighted Vote Method

In [14]:
ensemble_weighted_vote = EnsembleEfficientNet(models=models_list, method='weighted_vote', weights=weights)
ensemble_weighted_vote = ensemble_weighted_vote.to(device)

print(f"Evaluating ensemble with 'weighted_vote' method (weights: {weights})...")
response_weighted_vote = evaluation(ensemble_weighted_vote, test_loader, device)
result_weighted_vote = format_metrics(response_weighted_vote[0])
print("\n=== WEIGHTED VOTE METHOD ===")
print(result_weighted_vote)

⚠️ Normalizando pesos (soma atual: 0.7500)
Evaluating ensemble with 'weighted_vote' method (weights: [0.3, 0.25, 0.2])...


100%|██████████| 266/266 [09:24<00:00,  2.12s/it]



=== WEIGHTED VOTE METHOD ===
VAL_ACC      Mean: 19.724 | Std: 1.015 | 95% CI: [18.090, 21.420]
VAL_KAPPA    Mean: 0.022 | Std: 0.003 | 95% CI: [0.018, 0.027]
VAL_F1       Mean: 0.056 | Std: 0.002 | 95% CI: [0.052, 0.060]
VAL_RECALL   Mean: 0.131 | Std: 0.004 | 95% CI: [0.125, 0.137]
VAL_PRECISION Mean: 0.036 | Std: 0.002 | 95% CI: [0.033, 0.039]


## Results Comparison

In [15]:
# Extract kappa scores for comparison
import re

def extract_kappa(result_str):
    match = re.search(r'VAL_KAPPA\s+Mean: ([0-9.]+)', result_str)
    return float(match.group(1)) if match else 0.0

results_summary = {
    'max': extract_kappa(result_max),
    'mean': extract_kappa(result_mean),
    'weighted_mean': extract_kappa(result_weighted_mean),
    'majority_vote': extract_kappa(result_majority),
    'weighted_vote': extract_kappa(result_weighted_vote)
}

print("\n" + "="*50)
print("ENSEMBLE RESULTS COMPARISON (Kappa Score)")
print("="*50)
for method, kappa in sorted(results_summary.items(), key=lambda x: x[1], reverse=True):
    print(f"{method:20s}: {kappa:.4f}")

best_method = max(results_summary, key=results_summary.get)
print(f"\n✓ Best method: {best_method} (Kappa: {results_summary[best_method]:.4f})")


ENSEMBLE RESULTS COMPARISON (Kappa Score)
mean                : 0.8430
weighted_mean       : 0.8430
max                 : 0.8190
weighted_vote       : 0.0220
majority_vote       : 0.0000

✓ Best method: mean (Kappa: 0.8430)


## Optional: Test Different Temperature Values

Temperature controls the smoothness of probability distributions

In [16]:
temperatures = [0.5, 1.0, 2.0]
temp_results = {}

for temp in temperatures:
    ensemble_temp = EnsembleEfficientNet(models=models_list, method='weighted_vote', weights=weights, temperature=temp)
    ensemble_temp = ensemble_temp.to(device)
    
    print(f"\nEvaluating with temperature={temp}...")
    response_temp = evaluation(ensemble_temp, test_loader, device)
    result_temp = format_metrics(response_temp[0])
    temp_results[temp] = extract_kappa(result_temp)
    print(f"Temperature {temp}: Kappa = {temp_results[temp]:.4f}")

print("\n" + "="*50)
print("TEMPERATURE COMPARISON")
print("="*50)
for temp, kappa in sorted(temp_results.items(), key=lambda x: x[1], reverse=True):
    print(f"Temperature {temp:3.1f}: {kappa:.4f}")

⚠️ Normalizando pesos (soma atual: 0.7500)

Evaluating with temperature=0.5...


100%|██████████| 266/266 [09:25<00:00,  2.12s/it]


Temperature 0.5: Kappa = 0.0390
⚠️ Normalizando pesos (soma atual: 0.7500)

Evaluating with temperature=1.0...


 12%|█▏        | 31/266 [01:09<08:45,  2.24s/it]


KeyboardInterrupt: 